In [1]:
import xarray as xr

## Pangu

In [2]:
# model prediction data
pangu_surface = xr.open_dataset("../../AI_forecasting_result/Ragasa/Pangu/output_48/2025-09-21-19-00to2025-09-24-19-00/surface_combined.nc")
pangu_upper = xr.open_dataset("../../AI_forecasting_result/Ragasa/Pangu/output_48/2025-09-21-19-00to2025-09-24-19-00/upper_combined.nc")

### RMSE and ACC

This section computes latitude-weighted RMSE and ACC for Pangu against ERA5 at each 6-hour forecast lead from 6 h to 72 h.

ACC is calculated as a latitude-weighted anomaly correlation coefficient, using the mean ERA5 field over the 12 valid times as the climatology baseline. The results are saved to `./data/pangu_era5_rmse.csv` and `./data/pangu_era5_acc.csv`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PANGU_850_LEVEL = 3
EXPECTED_VALID_TIMES = pd.date_range("2025-09-22 01:00", periods=12, freq="6h")
OUTPUT_DIR = Path("./data")
RMSE_OUTPUT = OUTPUT_DIR / "pangu_era5_rmse.csv"
ACC_OUTPUT = OUTPUT_DIR / "pangu_era5_acc.csv"

VARIABLE_SPECS = [
    {"name": "U10", "layer": "surface", "var": "u10", "unit": "m s^-1"},
    {"name": "V10", "layer": "surface", "var": "v10", "unit": "m s^-1"},
    {"name": "MSLP", "layer": "surface", "var": "msl", "unit": "Pa"},
    {"name": "T2M", "layer": "surface", "var": "t2m", "unit": "K"},
    {"name": "U850", "layer": "upper", "var": "u", "unit": "m s^-1"},
    {"name": "V850", "layer": "upper", "var": "v", "unit": "m s^-1"},
    {"name": "Z850", "layer": "upper", "var": "z", "unit": "m^2 s^-2"},
    {"name": "T850", "layer": "upper", "var": "t", "unit": "K"},
]

def get_forecast_and_reference(spec):
    if spec["layer"] == "surface":
        forecast = pangu_surface[spec["var"]]
        reference = era_surface[spec["var"]]
    else:
        forecast = pangu_upper[spec["var"]].sel(level=PANGU_850_LEVEL)
        reference = era_850hPa[spec["var"]].sel(pressure_level=850, drop=True)

    forecast, reference = xr.align(forecast, reference, join="inner")
    forecast = forecast.transpose("valid_time", "latitude", "longitude")
    reference = reference.transpose("valid_time", "latitude", "longitude")
    return forecast, reference

def get_latitude_weights(latitudes):
    weights = np.cos(np.deg2rad(latitudes))
    return np.clip(weights, 0.0, None)

def weighted_rmse(forecast_field, reference_field, latitude_weights):
    weights_2d = np.broadcast_to(latitude_weights[:, None], forecast_field.shape)
    valid_mask = np.isfinite(forecast_field) & np.isfinite(reference_field)
    if not np.any(valid_mask):
        return np.nan

    weights_valid = weights_2d[valid_mask]
    squared_error = (forecast_field[valid_mask] - reference_field[valid_mask]) ** 2
    return np.sqrt(np.sum(weights_valid * squared_error) / np.sum(weights_valid))

def weighted_acc(forecast_field, reference_field, climatology_field, latitude_weights):
    forecast_anomaly = forecast_field - climatology_field
    reference_anomaly = reference_field - climatology_field
    weights_2d = np.broadcast_to(latitude_weights[:, None], forecast_field.shape)
    valid_mask = np.isfinite(forecast_anomaly) & np.isfinite(reference_anomaly)
    if not np.any(valid_mask):
        return np.nan

    weights_valid = weights_2d[valid_mask]
    forecast_valid = forecast_anomaly[valid_mask]
    reference_valid = reference_anomaly[valid_mask]
    denominator = np.sqrt(
        np.sum(weights_valid * forecast_valid ** 2) * np.sum(weights_valid * reference_valid ** 2)
    )
    if denominator == 0:
        return np.nan

    return np.sum(weights_valid * forecast_valid * reference_valid) / denominator

base_table = pd.DataFrame(
    {
        "forecast_hour": np.arange(6, 73, 6),
        "valid_time": EXPECTED_VALID_TIMES.strftime("%Y%m%d%H"),
    }
)
rmse_table = base_table.copy()
acc_table = base_table.copy()
units = {}

for spec in VARIABLE_SPECS:
    forecast_da, reference_da = get_forecast_and_reference(spec)
    valid_times = pd.to_datetime(forecast_da["valid_time"].values)
    if len(valid_times) != len(EXPECTED_VALID_TIMES) or not np.array_equal(valid_times.values, EXPECTED_VALID_TIMES.values):
        raise ValueError(
            f"Unexpected valid times for {spec['name']}: {valid_times.strftime('%Y%m%d%H').tolist()}"
        )

    climatology = reference_da.mean(dim="valid_time").values
    latitude_weights = get_latitude_weights(forecast_da["latitude"].values)
    rmse_values = []
    acc_values = []
    for time_index in range(forecast_da.sizes["valid_time"]):
        forecast_field = forecast_da.isel(valid_time=time_index).values
        reference_field = reference_da.isel(valid_time=time_index).values
        rmse_values.append(weighted_rmse(forecast_field, reference_field, latitude_weights))
        acc_values.append(weighted_acc(forecast_field, reference_field, climatology, latitude_weights))

    rmse_table[spec["name"]] = rmse_values
    acc_table[spec["name"]] = acc_values
    units[spec["name"]] = spec["unit"]

rmse_table.to_csv(RMSE_OUTPUT, index=False)
acc_table.to_csv(ACC_OUTPUT, index=False)

print("Units:")
for variable_name, unit in units.items():
    print(f"  {variable_name}: {unit}")

print(f"\nSaved RMSE results to: {RMSE_OUTPUT.resolve()}")
print(rmse_table.round(4))
print(f"\nSaved ACC results to: {ACC_OUTPUT.resolve()}")
print(acc_table.round(4))

Units:
  U10: m s^-1
  V10: m s^-1
  MSLP: Pa
  T2M: K
  U850: m s^-1
  V850: m s^-1
  Z850: m^2 s^-2
  T850: K

Saved RMSE results to: E:\CityU\Paper Code\06_AI_WRF_UCM\Figs\Fig3\data\pangu_era5_rmse.csv
    forecast_hour  valid_time     U10     V10        MSLP     T2M    U850  \
0               6  2025092201  0.6488  0.6608   41.932201  0.5869  1.0233   
1              12  2025092207  0.7345  0.7599   48.143101  0.6601  1.1539   
2              18  2025092213  0.9019  0.9287   61.937199  0.7248  1.3988   
3              24  2025092219  0.9758  1.0497   70.341904  0.8522  1.4991   
4              30  2025092301  1.1167  1.2114   84.866096  0.8694  1.7090   
5              36  2025092307  1.2124  1.2761   94.044403  0.9155  1.7882   
6              42  2025092313  1.3590  1.4171  114.477898  1.0213  2.0076   
7              48  2025092319  1.4661  1.5208  131.384598  1.1214  2.1691   
8              54  2025092401  1.5662  1.6511  142.754807  1.1728  2.3386   
9              60  202509

## GraphCast

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

if "EXPECTED_VALID_TIMES" not in globals():
    EXPECTED_VALID_TIMES = pd.date_range("2025-09-22 01:00", periods=12, freq="6h")

if "get_latitude_weights" not in globals():
    def get_latitude_weights(latitudes):
        weights = np.cos(np.deg2rad(latitudes))
        return np.clip(weights, 0.0, None)

if "weighted_rmse" not in globals():
    def weighted_rmse(forecast_field, reference_field, latitude_weights):
        weights_2d = np.broadcast_to(latitude_weights[:, None], forecast_field.shape)
        valid_mask = np.isfinite(forecast_field) & np.isfinite(reference_field)
        if not np.any(valid_mask):
            return np.nan

        weights_valid = weights_2d[valid_mask]
        squared_error = (forecast_field[valid_mask] - reference_field[valid_mask]) ** 2
        return np.sqrt(np.sum(weights_valid * squared_error) / np.sum(weights_valid))

if "weighted_acc" not in globals():
    def weighted_acc(forecast_field, reference_field, climatology_field, latitude_weights):
        forecast_anomaly = forecast_field - climatology_field
        reference_anomaly = reference_field - climatology_field
        weights_2d = np.broadcast_to(latitude_weights[:, None], forecast_field.shape)
        valid_mask = np.isfinite(forecast_anomaly) & np.isfinite(reference_anomaly)
        if not np.any(valid_mask):
            return np.nan

        weights_valid = weights_2d[valid_mask]
        forecast_valid = forecast_anomaly[valid_mask]
        reference_valid = reference_anomaly[valid_mask]
        denominator = np.sqrt(
            np.sum(weights_valid * forecast_valid ** 2) * np.sum(weights_valid * reference_valid ** 2)
        )
        if denominator == 0:
            return np.nan

        return np.sum(weights_valid * forecast_valid * reference_valid) / denominator

if "era_surface" not in globals():
    era_surface = xr.open_dataset("./data/era5_surface_2025092201-2025092419.nc")
if "era_850hPa" not in globals():
    era_850hPa = xr.open_dataset("./data/era5_850hPa_2025092201-2025092419.nc")

GRAPHCAST_DIR = Path("../../AI_forecasting_result/Ragasa/Graphcast/output_48/2025092119")
GRAPHCAST_850_LEVEL = 850
GRAPHCAST_RMSE_OUTPUT = Path("./data/graphcast_era5_rmse.csv")
GRAPHCAST_ACC_OUTPUT = Path("./data/graphcast_era5_acc.csv")

GRAPHCAST_VARIABLE_SPECS = [
    {"name": "U10", "layer": "surface", "graphcast_var": "10m_u_component_of_wind", "era_var": "u10", "unit": "m s^-1"},
    {"name": "V10", "layer": "surface", "graphcast_var": "10m_v_component_of_wind", "era_var": "v10", "unit": "m s^-1"},
    {"name": "MSLP", "layer": "surface", "graphcast_var": "mean_sea_level_pressure", "era_var": "msl", "unit": "Pa"},
    {"name": "T2M", "layer": "surface", "graphcast_var": "2m_temperature", "era_var": "t2m", "unit": "K"},
    {"name": "U850", "layer": "upper", "graphcast_var": "u_component_of_wind", "era_var": "u", "unit": "m s^-1"},
    {"name": "V850", "layer": "upper", "graphcast_var": "v_component_of_wind", "era_var": "v", "unit": "m s^-1"},
    {"name": "Z850", "layer": "upper", "graphcast_var": "geopotential", "era_var": "z", "unit": "m^2 s^-2"},
    {"name": "T850", "layer": "upper", "graphcast_var": "temperature", "era_var": "t", "unit": "K"},
]

graphcast_files = sorted(
    GRAPHCAST_DIR.glob("gc_operational_predict_data_*.nc"),
    key=lambda path: int(path.stem.rsplit("_", 1)[-1]),
)
if len(graphcast_files) != len(EXPECTED_VALID_TIMES):
    raise ValueError(f"Expected {len(EXPECTED_VALID_TIMES)} GraphCast files, found {len(graphcast_files)}")

def load_graphcast_forecast(spec):
    forecast_slices = []
    for path in graphcast_files:
        with xr.open_dataset(path, decode_timedelta=False) as ds:
            valid_time = pd.Timestamp(np.asarray(ds["datetime"].values).reshape(-1)[0])
            forecast = ds[spec["graphcast_var"]]
            if "batch" in forecast.dims:
                forecast = forecast.isel(batch=0, drop=True)
            if "time" in forecast.dims:
                forecast = forecast.isel(time=0, drop=True)
            if spec["layer"] == "upper":
                forecast = forecast.sel(level=GRAPHCAST_850_LEVEL, drop=True)

            forecast = forecast.rename({"lat": "latitude", "lon": "longitude"})
            forecast = forecast.expand_dims(valid_time=[valid_time]).load()
            forecast_slices.append(forecast)

    return xr.concat(forecast_slices, dim="valid_time").sortby("valid_time")

def get_graphcast_forecast_and_reference(spec):
    forecast = load_graphcast_forecast(spec)
    if spec["layer"] == "surface":
        reference = era_surface[spec["era_var"]]
    else:
        reference = era_850hPa[spec["era_var"]].sel(pressure_level=850, drop=True)

    reference = reference.sortby("valid_time")
    forecast = forecast.reindex(
        latitude=reference["latitude"],
        longitude=reference["longitude"],
    )
    forecast, reference = xr.align(forecast, reference, join="inner")
    forecast = forecast.transpose("valid_time", "latitude", "longitude")
    reference = reference.transpose("valid_time", "latitude", "longitude")
    return forecast, reference

graphcast_base_table = pd.DataFrame(
    {
        "forecast_hour": np.arange(6, 73, 6),
        "valid_time": EXPECTED_VALID_TIMES.strftime("%Y%m%d%H"),
    }
)
graphcast_rmse_table = graphcast_base_table.copy()
graphcast_acc_table = graphcast_base_table.copy()
graphcast_units = {}

for spec in GRAPHCAST_VARIABLE_SPECS:
    forecast_da, reference_da = get_graphcast_forecast_and_reference(spec)
    valid_times = pd.to_datetime(forecast_da["valid_time"].values)
    if len(valid_times) != len(EXPECTED_VALID_TIMES) or not np.array_equal(valid_times.values, EXPECTED_VALID_TIMES.values):
        raise ValueError(
            f"Unexpected valid times for {spec['name']}: {valid_times.strftime('%Y%m%d%H').tolist()}"
        )

    climatology = reference_da.mean(dim="valid_time").values
    latitude_weights = get_latitude_weights(forecast_da["latitude"].values)
    rmse_values = []
    acc_values = []
    for time_index in range(forecast_da.sizes["valid_time"]):
        forecast_field = forecast_da.isel(valid_time=time_index).values
        reference_field = reference_da.isel(valid_time=time_index).values
        rmse_values.append(weighted_rmse(forecast_field, reference_field, latitude_weights))
        acc_values.append(weighted_acc(forecast_field, reference_field, climatology, latitude_weights))

    graphcast_rmse_table[spec["name"]] = rmse_values
    graphcast_acc_table[spec["name"]] = acc_values
    graphcast_units[spec["name"]] = spec["unit"]

graphcast_rmse_table.to_csv(GRAPHCAST_RMSE_OUTPUT, index=False)
graphcast_acc_table.to_csv(GRAPHCAST_ACC_OUTPUT, index=False)

print("Units:")
for variable_name, unit in graphcast_units.items():
    print(f"  {variable_name}: {unit}")

print(f"\nSaved RMSE results to: {GRAPHCAST_RMSE_OUTPUT.resolve()}")
print(graphcast_rmse_table.round(4))
print(f"\nSaved ACC results to: {GRAPHCAST_ACC_OUTPUT.resolve()}")
print(graphcast_acc_table.round(4))

Units:
  U10: m s^-1
  V10: m s^-1
  MSLP: Pa
  T2M: K
  U850: m s^-1
  V850: m s^-1
  Z850: m^2 s^-2
  T850: K

Saved RMSE results to: E:\CityU\Paper Code\06_AI_WRF_UCM\Figs\Fig3\data\graphcast_era5_rmse.csv
    forecast_hour  valid_time     U10     V10      MSLP     T2M    U850  \
0               6  2025092201  0.6503  0.6508   43.9189  0.7688  0.9446   
1              12  2025092207  0.7653  0.7716   50.1793  0.8183  1.0816   
2              18  2025092213  0.8728  0.9026   60.2531  0.8583  1.2911   
3              24  2025092219  0.9478  0.9836   66.0763  0.8282  1.3816   
4              30  2025092301  1.0502  1.0963   78.9122  0.9565  1.5466   
5              36  2025092307  1.1285  1.1678   88.4993  0.9245  1.6154   
6              42  2025092313  1.2416  1.2652  101.4382  1.0183  1.7728   
7              48  2025092319  1.3162  1.3517  115.1498  0.9913  1.9043   
8              54  2025092401  1.3947  1.4451  124.9191  1.0593  2.0782   
9              60  2025092407  1.5481  1.

## FengWu

In [18]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

if "EXPECTED_VALID_TIMES" not in globals():
    EXPECTED_VALID_TIMES = pd.date_range("2025-09-22 01:00", periods=12, freq="6h")

if "get_latitude_weights" not in globals():
    def get_latitude_weights(latitudes):
        weights = np.cos(np.deg2rad(latitudes))
        return np.clip(weights, 0.0, None)

if "weighted_rmse" not in globals():
    def weighted_rmse(forecast_field, reference_field, latitude_weights):
        weights_2d = np.broadcast_to(latitude_weights[:, None], forecast_field.shape)
        valid_mask = np.isfinite(forecast_field) & np.isfinite(reference_field)
        if not np.any(valid_mask):
            return np.nan

        weights_valid = weights_2d[valid_mask]
        squared_error = (forecast_field[valid_mask] - reference_field[valid_mask]) ** 2
        return np.sqrt(np.sum(weights_valid * squared_error) / np.sum(weights_valid))

if "weighted_acc" not in globals():
    def weighted_acc(forecast_field, reference_field, climatology_field, latitude_weights):
        forecast_anomaly = forecast_field - climatology_field
        reference_anomaly = reference_field - climatology_field
        weights_2d = np.broadcast_to(latitude_weights[:, None], forecast_field.shape)
        valid_mask = np.isfinite(forecast_anomaly) & np.isfinite(reference_anomaly)
        if not np.any(valid_mask):
            return np.nan

        weights_valid = weights_2d[valid_mask]
        forecast_valid = forecast_anomaly[valid_mask]
        reference_valid = reference_anomaly[valid_mask]
        denominator = np.sqrt(
            np.sum(weights_valid * forecast_valid ** 2) * np.sum(weights_valid * reference_valid ** 2)
        )
        if denominator == 0:
            return np.nan

        return np.sum(weights_valid * forecast_valid * reference_valid) / denominator

if "era_surface" not in globals():
    era_surface = xr.open_dataset("./data/era5_surface_2025092201-2025092419.nc")
if "era_850hPa" not in globals():
    era_850hPa = xr.open_dataset("./data/era5_850hPa_2025092201-2025092419.nc")

FENGWU_DIR = Path("../../AI_forecasting_result/Ragasa/Fengwu/output_48/2025-09-21-13-00_to_2025-09-21-19-00")
FENGWU_RMSE_OUTPUT = Path("./data/fengwu_era5_rmse.csv")
FENGWU_ACC_OUTPUT = Path("./data/fengwu_era5_acc.csv")

FENGWU_VARIABLE_SPECS = [
    {"name": "U10", "layer": "surface", "fengwu_var": "u10", "era_var": "u10", "unit": "m s^-1"},
    {"name": "V10", "layer": "surface", "fengwu_var": "v10", "era_var": "v10", "unit": "m s^-1"},
    {"name": "MSLP", "layer": "surface", "fengwu_var": "msl", "era_var": "msl", "unit": "Pa"},
    {"name": "T2M", "layer": "surface", "fengwu_var": "t2m", "era_var": "t2m", "unit": "K"},
    {"name": "U850", "layer": "upper", "fengwu_var": "u850", "era_var": "u", "unit": "m s^-1"},
    {"name": "V850", "layer": "upper", "fengwu_var": "v850", "era_var": "v", "unit": "m s^-1"},
    {"name": "Z850", "layer": "upper", "fengwu_var": "z850", "era_var": "z", "unit": "m^2 s^-2"},
    {"name": "T850", "layer": "upper", "fengwu_var": "t850", "era_var": "t", "unit": "K"},
]

fengwu_files = sorted(
    FENGWU_DIR.glob("decoded_output_*.nc"),
    key=lambda path: int(path.stem.rsplit("_", 1)[-1]),
)
if len(fengwu_files) != len(EXPECTED_VALID_TIMES):
    raise ValueError(f"Expected {len(EXPECTED_VALID_TIMES)} Fengwu files, found {len(fengwu_files)}")


def load_fengwu_forecast(spec):
    forecast_slices = []
    for path in fengwu_files:
        with xr.open_dataset(path) as ds:
            forecast = ds[spec["fengwu_var"]]
            valid_times = pd.to_datetime(forecast["valid_time"].values)
            if len(valid_times) != 1:
                raise ValueError(f"Expected exactly one valid time in {path.name}, found {len(valid_times)}")

            forecast = forecast.load()
            forecast_slices.append(forecast)

    return xr.concat(forecast_slices, dim="valid_time").sortby("valid_time")


def get_fengwu_forecast_and_reference(spec):
    forecast = load_fengwu_forecast(spec)
    if spec["layer"] == "surface":
        reference = era_surface[spec["era_var"]]
    else:
        reference = era_850hPa[spec["era_var"]].sel(pressure_level=850, drop=True)

    reference = reference.sortby("valid_time")
    forecast, reference = xr.align(forecast, reference, join="inner")
    forecast = forecast.transpose("valid_time", "latitude", "longitude")
    reference = reference.transpose("valid_time", "latitude", "longitude")
    return forecast, reference


fengwu_base_table = pd.DataFrame(
    {
        "forecast_hour": np.arange(6, 73, 6),
        "valid_time": EXPECTED_VALID_TIMES.strftime("%Y%m%d%H"),
    }
)

fengwu_rmse_table = fengwu_base_table.copy()
fengwu_acc_table = fengwu_base_table.copy()
fengwu_units = {}

for spec in FENGWU_VARIABLE_SPECS:
    forecast_da, reference_da = get_fengwu_forecast_and_reference(spec)
    valid_times = pd.to_datetime(forecast_da["valid_time"].values)
    if len(valid_times) != len(EXPECTED_VALID_TIMES) or not np.array_equal(valid_times.values, EXPECTED_VALID_TIMES.values):
        raise ValueError(
            f"Unexpected valid times for {spec['name']}: {valid_times.strftime('%Y%m%d%H').tolist()}"
        )

    climatology = reference_da.mean(dim="valid_time").values
    latitude_weights = get_latitude_weights(forecast_da["latitude"].values)
    rmse_values = []
    acc_values = []

    for time_index in range(forecast_da.sizes["valid_time"]):
        forecast_field = forecast_da.isel(valid_time=time_index).values
        reference_field = reference_da.isel(valid_time=time_index).values
        rmse_values.append(weighted_rmse(forecast_field, reference_field, latitude_weights))
        acc_values.append(weighted_acc(forecast_field, reference_field, climatology, latitude_weights))

    fengwu_rmse_table[spec["name"]] = rmse_values
    fengwu_acc_table[spec["name"]] = acc_values
    fengwu_units[spec["name"]] = spec["unit"]

fengwu_rmse_table.to_csv(FENGWU_RMSE_OUTPUT, index=False)
fengwu_acc_table.to_csv(FENGWU_ACC_OUTPUT, index=False)

print("Units:")
for variable_name, unit in fengwu_units.items():
    print(f"  {variable_name}: {unit}")

print(f"\nSaved RMSE results to: {FENGWU_RMSE_OUTPUT.resolve()}")
print(fengwu_rmse_table.round(4))
print(f"\nSaved ACC results to: {FENGWU_ACC_OUTPUT.resolve()}")
print(fengwu_acc_table.round(4))

Units:
  U10: m s^-1
  V10: m s^-1
  MSLP: Pa
  T2M: K
  U850: m s^-1
  V850: m s^-1
  Z850: m^2 s^-2
  T850: K

Saved RMSE results to: E:\CityU\Paper Code\06_AI_WRF_UCM\Figs\Fig3\data\fengwu_era5_rmse.csv
    forecast_hour  valid_time     U10     V10        MSLP     T2M    U850  \
0               6  2025092201  0.6265  0.6357   41.040401  0.6034  0.9542   
1              12  2025092207  0.6911  0.7136   45.849899  0.7480  1.0454   
2              18  2025092213  0.8406  0.8586   56.033901  0.7508  1.2731   
3              24  2025092219  0.8923  0.9262   61.080299  0.7710  1.3510   
4              30  2025092301  1.0134  1.0569   72.317398  0.8242  1.5416   
5              36  2025092307  1.0825  1.1065   81.367599  0.9094  1.5849   
6              42  2025092313  1.1973  1.2212   93.135902  0.9377  1.7411   
7              48  2025092319  1.2612  1.3156  103.559601  0.9639  1.8447   
8              54  2025092401  1.3222  1.4023  112.953400  0.9793  1.9640   
9              60  20250

## FuXi

In [22]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

if "EXPECTED_VALID_TIMES" not in globals():
    EXPECTED_VALID_TIMES = pd.date_range("2025-09-22 01:00", periods=12, freq="6h")

if "get_latitude_weights" not in globals():
    def get_latitude_weights(latitudes):
        weights = np.cos(np.deg2rad(latitudes))
        return np.clip(weights, 0.0, None)

if "weighted_rmse" not in globals():
    def weighted_rmse(forecast_field, reference_field, latitude_weights):
        weights_2d = np.broadcast_to(latitude_weights[:, None], forecast_field.shape)
        valid_mask = np.isfinite(forecast_field) & np.isfinite(reference_field)
        if not np.any(valid_mask):
            return np.nan

        weights_valid = weights_2d[valid_mask]
        squared_error = (forecast_field[valid_mask] - reference_field[valid_mask]) ** 2
        return np.sqrt(np.sum(weights_valid * squared_error) / np.sum(weights_valid))

if "weighted_acc" not in globals():
    def weighted_acc(forecast_field, reference_field, climatology_field, latitude_weights):
        forecast_anomaly = forecast_field - climatology_field
        reference_anomaly = reference_field - climatology_field
        weights_2d = np.broadcast_to(latitude_weights[:, None], forecast_field.shape)
        valid_mask = np.isfinite(forecast_anomaly) & np.isfinite(reference_anomaly)
        if not np.any(valid_mask):
            return np.nan

        weights_valid = weights_2d[valid_mask]
        forecast_valid = forecast_anomaly[valid_mask]
        reference_valid = reference_anomaly[valid_mask]
        denominator = np.sqrt(
            np.sum(weights_valid * forecast_valid ** 2) * np.sum(weights_valid * reference_valid ** 2)
        )
        if denominator == 0:
            return np.nan

        return np.sum(weights_valid * forecast_valid * reference_valid) / denominator

if "era_surface" not in globals():
    era_surface = xr.open_dataset("./data/era5_surface_2025092201-2025092419.nc")
if "era_850hPa" not in globals():
    era_850hPa = xr.open_dataset("./data/era5_850hPa_2025092201-2025092419.nc")

FUXI_DIR = Path("../../AI_forecasting_result/Ragasa/Fuxi/output_48/2025092113_to_2025092119")
FUXI_INIT_TIME = pd.Timestamp("2025-09-21 19:00")
FUXI_RMSE_OUTPUT = Path("./data/fuxi_era5_rmse.csv")
FUXI_ACC_OUTPUT = Path("./data/fuxi_era5_acc.csv")

FUXI_VARIABLE_SPECS = [
    {"name": "U10", "layer": "surface", "fuxi_level": "U10", "era_var": "u10", "unit": "m s^-1"},
    {"name": "V10", "layer": "surface", "fuxi_level": "V10", "era_var": "v10", "unit": "m s^-1"},
    {"name": "MSLP", "layer": "surface", "fuxi_level": "MSL", "era_var": "msl", "unit": "Pa"},
    {"name": "T2M", "layer": "surface", "fuxi_level": "T2M", "era_var": "t2m", "unit": "K"},
    {"name": "U850", "layer": "upper", "fuxi_level": "U850", "era_var": "u", "unit": "m s^-1"},
    {"name": "V850", "layer": "upper", "fuxi_level": "V850", "era_var": "v", "unit": "m s^-1"},
    {"name": "Z850", "layer": "upper", "fuxi_level": "Z850", "era_var": "z", "unit": "m^2 s^-2"},
    {"name": "T850", "layer": "upper", "fuxi_level": "T850", "era_var": "t", "unit": "K"},
]

fuxi_files = sorted(FUXI_DIR.glob("*.nc"), key=lambda path: int(path.stem))
if len(fuxi_files) != len(EXPECTED_VALID_TIMES):
    raise ValueError(f"Expected {len(EXPECTED_VALID_TIMES)} Fuxi files, found {len(fuxi_files)}")


def load_fuxi_forecast(spec):
    forecast_slices = []
    for path in fuxi_files:
        lead_hour = int(path.stem)
        valid_time = FUXI_INIT_TIME + pd.Timedelta(hours=lead_hour)
        with xr.open_dataset(path, decode_timedelta=False) as ds:
            forecast = ds["__xarray_dataarray_variable__"].sel(level=spec["fuxi_level"])
            if "time" in forecast.dims:
                forecast = forecast.isel(time=0, drop=True)
            if "step" in forecast.dims:
                forecast = forecast.isel(step=0, drop=True)

            forecast = forecast.rename({"lat": "latitude", "lon": "longitude"})
            forecast = forecast.expand_dims(valid_time=[valid_time]).load()
            forecast_slices.append(forecast)

    return xr.concat(forecast_slices, dim="valid_time").sortby("valid_time")


def get_fuxi_forecast_and_reference(spec):
    forecast = load_fuxi_forecast(spec)
    if spec["layer"] == "surface":
        reference = era_surface[spec["era_var"]]
    else:
        reference = era_850hPa[spec["era_var"]].sel(pressure_level=850, drop=True)

    reference = reference.sortby("valid_time")
    forecast = forecast.reindex(
        latitude=reference["latitude"],
        longitude=reference["longitude"],
    )
    forecast, reference = xr.align(forecast, reference, join="inner")
    forecast = forecast.transpose("valid_time", "latitude", "longitude")
    reference = reference.transpose("valid_time", "latitude", "longitude")
    return forecast, reference


fuxi_base_table = pd.DataFrame(
    {
        "forecast_hour": np.arange(6, 73, 6),
        "valid_time": EXPECTED_VALID_TIMES.strftime("%Y%m%d%H"),
    }
)

fuxi_rmse_table = fuxi_base_table.copy()
fuxi_acc_table = fuxi_base_table.copy()
fuxi_units = {}

for spec in FUXI_VARIABLE_SPECS:
    forecast_da, reference_da = get_fuxi_forecast_and_reference(spec)
    valid_times = pd.to_datetime(forecast_da["valid_time"].values)
    if len(valid_times) != len(EXPECTED_VALID_TIMES) or not np.array_equal(valid_times.values, EXPECTED_VALID_TIMES.values):
        raise ValueError(
            f"Unexpected valid times for {spec['name']}: {valid_times.strftime('%Y%m%d%H').tolist()}"
        )

    climatology = reference_da.mean(dim="valid_time").values
    latitude_weights = get_latitude_weights(forecast_da["latitude"].values)
    rmse_values = []
    acc_values = []

    for time_index in range(forecast_da.sizes["valid_time"]):
        forecast_field = forecast_da.isel(valid_time=time_index).values
        reference_field = reference_da.isel(valid_time=time_index).values
        rmse_values.append(weighted_rmse(forecast_field, reference_field, latitude_weights))
        acc_values.append(weighted_acc(forecast_field, reference_field, climatology, latitude_weights))

    fuxi_rmse_table[spec["name"]] = rmse_values
    fuxi_acc_table[spec["name"]] = acc_values
    fuxi_units[spec["name"]] = spec["unit"]

fuxi_rmse_table.to_csv(FUXI_RMSE_OUTPUT, index=False)
fuxi_acc_table.to_csv(FUXI_ACC_OUTPUT, index=False)

print("Units:")
for variable_name, unit in fuxi_units.items():
    print(f"  {variable_name}: {unit}")

print(f"\nSaved RMSE results to: {FUXI_RMSE_OUTPUT.resolve()}")
print(fuxi_rmse_table.round(4))
print(f"\nSaved ACC results to: {FUXI_ACC_OUTPUT.resolve()}")
print(fuxi_acc_table.round(4))

Units:
  U10: m s^-1
  V10: m s^-1
  MSLP: Pa
  T2M: K
  U850: m s^-1
  V850: m s^-1
  Z850: m^2 s^-2
  T850: K

Saved RMSE results to: E:\CityU\Paper Code\06_AI_WRF_UCM\Figs\Fig3\data\fuxi_era5_rmse.csv
    forecast_hour  valid_time     U10     V10      MSLP     T2M    U850  \
0               6  2025092201  0.6443  0.6562   46.7846  0.7475  0.9673   
1              12  2025092207  0.7086  0.7260   51.1321  0.8541  1.0663   
2              18  2025092213  0.8404  0.8611   58.4233  0.7407  1.2897   
3              24  2025092219  0.9000  0.9393   67.2033  0.7634  1.3666   
4              30  2025092301  1.0239  1.0641   78.3450  0.8619  1.5586   
5              36  2025092307  1.1122  1.1414   92.0253  0.9478  1.6327   
6              42  2025092313  1.2157  1.2562   96.7548  0.9155  1.7774   
7              48  2025092319  1.2984  1.3423  113.2783  0.9576  1.8961   
8              54  2025092401  1.3582  1.4394  118.5415  1.0063  2.0063   
9              60  2025092407  1.4583  1.5254 

## Aurora

In [24]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

if "EXPECTED_VALID_TIMES" not in globals():
    EXPECTED_VALID_TIMES = pd.date_range("2025-09-22 01:00", periods=12, freq="6h")

if "get_latitude_weights" not in globals():
    def get_latitude_weights(latitudes):
        weights = np.cos(np.deg2rad(latitudes))
        return np.clip(weights, 0.0, None)

if "weighted_rmse" not in globals():
    def weighted_rmse(forecast_field, reference_field, latitude_weights):
        weights_2d = np.broadcast_to(latitude_weights[:, None], forecast_field.shape)
        valid_mask = np.isfinite(forecast_field) & np.isfinite(reference_field)
        if not np.any(valid_mask):
            return np.nan

        weights_valid = weights_2d[valid_mask]
        squared_error = (forecast_field[valid_mask] - reference_field[valid_mask]) ** 2
        return np.sqrt(np.sum(weights_valid * squared_error) / np.sum(weights_valid))

if "weighted_acc" not in globals():
    def weighted_acc(forecast_field, reference_field, climatology_field, latitude_weights):
        forecast_anomaly = forecast_field - climatology_field
        reference_anomaly = reference_field - climatology_field
        weights_2d = np.broadcast_to(latitude_weights[:, None], forecast_field.shape)
        valid_mask = np.isfinite(forecast_anomaly) & np.isfinite(reference_anomaly)
        if not np.any(valid_mask):
            return np.nan

        weights_valid = weights_2d[valid_mask]
        forecast_valid = forecast_anomaly[valid_mask]
        reference_valid = reference_anomaly[valid_mask]
        denominator = np.sqrt(
            np.sum(weights_valid * forecast_valid ** 2) * np.sum(weights_valid * reference_valid ** 2)
        )
        if denominator == 0:
            return np.nan

        return np.sum(weights_valid * forecast_valid * reference_valid) / denominator

if "era_surface" not in globals():
    era_surface = xr.open_dataset("./data/era5_surface_2025092201-2025092419.nc")
if "era_850hPa" not in globals():
    era_850hPa = xr.open_dataset("./data/era5_850hPa_2025092201-2025092419.nc")

AURORA_DIR = Path("../../AI_forecasting_result/Ragasa/Aurora/output_48/WP/Ragasa/202509211300")
AURORA_RMSE_OUTPUT = Path("./data/aurora_era5_rmse.csv")
AURORA_ACC_OUTPUT = Path("./data/aurora_era5_acc.csv")

AURORA_VARIABLE_SPECS = [
    {"name": "U10", "layer": "surface", "aurora_var": "u10", "era_var": "u10", "unit": "m s^-1"},
    {"name": "V10", "layer": "surface", "aurora_var": "v10", "era_var": "v10", "unit": "m s^-1"},
    {"name": "MSLP", "layer": "surface", "aurora_var": "msl", "era_var": "msl", "unit": "Pa"},
    {"name": "T2M", "layer": "surface", "aurora_var": "t2m", "era_var": "t2m", "unit": "K"},
    {"name": "U850", "layer": "upper", "aurora_var": "u", "era_var": "u", "unit": "m s^-1"},
    {"name": "V850", "layer": "upper", "aurora_var": "v", "era_var": "v", "unit": "m s^-1"},
    {"name": "Z850", "layer": "upper", "aurora_var": "z", "era_var": "z", "unit": "m^2 s^-2"},
    {"name": "T850", "layer": "upper", "aurora_var": "t", "era_var": "t", "unit": "K"},
]

aurora_files = sorted(AURORA_DIR.glob("*.nc"))
if len(aurora_files) != len(EXPECTED_VALID_TIMES):
    raise ValueError(f"Expected {len(EXPECTED_VALID_TIMES)} Aurora files, found {len(aurora_files)}")


def load_aurora_forecast(spec):
    forecast_slices = []
    for path in aurora_files:
        with xr.open_dataset(path, decode_timedelta=False) as ds:
            valid_time = pd.Timestamp(np.asarray(ds["time"].values).reshape(-1)[0])
            forecast = ds[spec["aurora_var"]]
            if spec["layer"] == "upper":
                forecast = forecast.sel(level=850, drop=True)
            if "time" in forecast.dims:
                forecast = forecast.isel(time=0, drop=True)

            forecast = forecast.rename({"lat": "latitude", "lon": "longitude"})
            forecast = forecast.expand_dims(valid_time=[valid_time]).load()
            forecast_slices.append(forecast)

    return xr.concat(forecast_slices, dim="valid_time").sortby("valid_time")


def get_aurora_forecast_and_reference(spec):
    forecast = load_aurora_forecast(spec)
    if spec["layer"] == "surface":
        reference = era_surface[spec["era_var"]]
    else:
        reference = era_850hPa[spec["era_var"]].sel(pressure_level=850, drop=True)

    reference = reference.sortby("valid_time")
    forecast = forecast.reindex(
        latitude=reference["latitude"],
        longitude=reference["longitude"],
    )
    forecast, reference = xr.align(forecast, reference, join="inner")
    forecast = forecast.transpose("valid_time", "latitude", "longitude")
    reference = reference.transpose("valid_time", "latitude", "longitude")
    return forecast, reference


aurora_base_table = pd.DataFrame(
    {
        "forecast_hour": np.arange(6, 73, 6),
        "valid_time": EXPECTED_VALID_TIMES.strftime("%Y%m%d%H"),
    }
)

aurora_rmse_table = aurora_base_table.copy()
aurora_acc_table = aurora_base_table.copy()
aurora_units = {}

for spec in AURORA_VARIABLE_SPECS:
    forecast_da, reference_da = get_aurora_forecast_and_reference(spec)
    valid_times = pd.to_datetime(forecast_da["valid_time"].values)
    if len(valid_times) != len(EXPECTED_VALID_TIMES) or not np.array_equal(valid_times.values, EXPECTED_VALID_TIMES.values):
        raise ValueError(
            f"Unexpected valid times for {spec['name']}: {valid_times.strftime('%Y%m%d%H').tolist()}"
        )

    climatology = reference_da.mean(dim="valid_time").values
    latitude_weights = get_latitude_weights(forecast_da["latitude"].values)
    rmse_values = []
    acc_values = []

    for time_index in range(forecast_da.sizes["valid_time"]):
        forecast_field = forecast_da.isel(valid_time=time_index).values
        reference_field = reference_da.isel(valid_time=time_index).values
        rmse_values.append(weighted_rmse(forecast_field, reference_field, latitude_weights))
        acc_values.append(weighted_acc(forecast_field, reference_field, climatology, latitude_weights))

    aurora_rmse_table[spec["name"]] = rmse_values
    aurora_acc_table[spec["name"]] = acc_values
    aurora_units[spec["name"]] = spec["unit"]

aurora_rmse_table.to_csv(AURORA_RMSE_OUTPUT, index=False)
aurora_acc_table.to_csv(AURORA_ACC_OUTPUT, index=False)

print("Units:")
for variable_name, unit in aurora_units.items():
    print(f"  {variable_name}: {unit}")

print(f"\nSaved RMSE results to: {AURORA_RMSE_OUTPUT.resolve()}")
print(aurora_rmse_table.round(4))
print(f"\nSaved ACC results to: {AURORA_ACC_OUTPUT.resolve()}")
print(aurora_acc_table.round(4))

Units:
  U10: m s^-1
  V10: m s^-1
  MSLP: Pa
  T2M: K
  U850: m s^-1
  V850: m s^-1
  Z850: m^2 s^-2
  T850: K

Saved RMSE results to: E:\CityU\Paper Code\06_AI_WRF_UCM\Figs\Fig3\data\aurora_era5_rmse.csv
    forecast_hour  valid_time     U10     V10      MSLP     T2M    U850  \
0               6  2025092201  0.5627  0.5701   37.2491  0.5070  0.8714   
1              12  2025092207  0.6320  0.6696   42.6020  0.5580  0.9728   
2              18  2025092213  0.7882  0.8208   52.4248  0.6219  1.2224   
3              24  2025092219  0.8764  0.9225   59.7611  0.7242  1.3211   
4              30  2025092301  1.0177  1.0918   75.7359  0.7761  1.5590   
5              36  2025092307  1.1186  1.1635   83.7674  0.7973  1.6432   
6              42  2025092313  1.2579  1.3060   96.1484  0.9162  1.8457   
7              48  2025092319  1.3566  1.4111  109.5229  0.9942  1.9984   
8              54  2025092401  1.4383  1.5190  119.7100  1.0530  2.1400   
9              60  2025092407  1.5603  1.581

## Regional

In [1]:
# Regional RMSE/ACC on the minimal 0.25° subdomain covering d01_range
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

d01_range = (9.733, 33.868, 97.149, 131.277)  # (lat_min, lat_max, lon_min, lon_max)
REGION_TAG = "d01"
GRID_DEG = 0.25
OUTPUT_DIR = Path("./data")

if "EXPECTED_VALID_TIMES" not in globals():
    EXPECTED_VALID_TIMES = pd.date_range("2025-09-22 01:00", periods=12, freq="6h")

# Ensure ERA5 is available (global code loads it in other sections; keep this cell runnable standalone).
if "era_surface" not in globals():
    era_surface = xr.open_dataset("./data/era5_surface_2025092201-2025092419.nc")
if "era_850hPa" not in globals():
    era_850hPa = xr.open_dataset("./data/era5_850hPa_2025092201-2025092419.nc")

if "get_latitude_weights" not in globals():
    def get_latitude_weights(latitudes):
        weights = np.cos(np.deg2rad(latitudes))
        return np.clip(weights, 0.0, None)

if "weighted_rmse" not in globals():
    def weighted_rmse(forecast_field, reference_field, latitude_weights):
        weights_2d = np.broadcast_to(latitude_weights[:, None], forecast_field.shape)
        valid_mask = np.isfinite(forecast_field) & np.isfinite(reference_field)
        if not np.any(valid_mask):
            return np.nan

        weights_valid = weights_2d[valid_mask]
        squared_error = (forecast_field[valid_mask] - reference_field[valid_mask]) ** 2
        return np.sqrt(np.sum(weights_valid * squared_error) / np.sum(weights_valid))

if "weighted_acc" not in globals():
    def weighted_acc(forecast_field, reference_field, climatology_field, latitude_weights):
        forecast_anomaly = forecast_field - climatology_field
        reference_anomaly = reference_field - climatology_field
        weights_2d = np.broadcast_to(latitude_weights[:, None], forecast_field.shape)
        valid_mask = np.isfinite(forecast_anomaly) & np.isfinite(reference_anomaly)
        if not np.any(valid_mask):
            return np.nan

        weights_valid = weights_2d[valid_mask]
        forecast_valid = forecast_anomaly[valid_mask]
        reference_valid = reference_anomaly[valid_mask]
        denominator = np.sqrt(
            np.sum(weights_valid * forecast_valid**2) * np.sum(weights_valid * reference_valid**2),
        )
        if denominator == 0:
            return np.nan

        return np.sum(weights_valid * forecast_valid * reference_valid) / denominator


def _find_leq(sorted_values, value):
    idx = int(np.searchsorted(sorted_values, value, side="right") - 1)
    idx = max(0, min(idx, len(sorted_values) - 1))
    return float(sorted_values[idx])


def _find_geq(sorted_values, value):
    idx = int(np.searchsorted(sorted_values, value, side="left"))
    idx = max(0, min(idx, len(sorted_values) - 1))
    return float(sorted_values[idx])


def minimal_covering_slices(era_latitudes, era_longitudes, lat_min, lat_max, lon_min, lon_max, step_deg=0.25):
    lat_values = np.asarray(era_latitudes, dtype=float)
    lon_values = np.asarray(era_longitudes, dtype=float)

    lat_ascending = lat_values[0] < lat_values[-1]
    lon_ascending = lon_values[0] < lon_values[-1]
    lat_sorted = lat_values if lat_ascending else lat_values[::-1]
    lon_sorted = lon_values if lon_ascending else lon_values[::-1]

    lat_min_snap = np.floor(lat_min / step_deg) * step_deg
    lat_max_snap = np.ceil(lat_max / step_deg) * step_deg
    lon_min_snap = np.floor(lon_min / step_deg) * step_deg
    lon_max_snap = np.ceil(lon_max / step_deg) * step_deg

    lat_lower = _find_leq(lat_sorted, lat_min_snap)
    lat_upper = _find_geq(lat_sorted, lat_max_snap)
    lon_lower = _find_leq(lon_sorted, lon_min_snap)
    lon_upper = _find_geq(lon_sorted, lon_max_snap)

    if lat_ascending:
        lat_slice = slice(lat_lower, lat_upper)
    else:
        lat_slice = slice(lat_upper, lat_lower)

    if lon_ascending:
        lon_slice = slice(lon_lower, lon_upper)
    else:
        lon_slice = slice(lon_upper, lon_lower)

    return {
        "snapped": (float(lat_min_snap), float(lat_max_snap), float(lon_min_snap), float(lon_max_snap)),
        "bounds": (lat_lower, lat_upper, lon_lower, lon_upper),
        "lat_slice": lat_slice,
        "lon_slice": lon_slice,
        "lat_ascending": lat_ascending,
        "lon_ascending": lon_ascending,
    }


lat_min, lat_max, lon_min, lon_max = d01_range
region_info = minimal_covering_slices(
    era_surface["latitude"].values,
    era_surface["longitude"].values,
    lat_min,
    lat_max,
    lon_min,
    lon_max,
    step_deg=GRID_DEG,
)

REGION_LAT_SLICE = region_info["lat_slice"]
REGION_LON_SLICE = region_info["lon_slice"]

ERA_SURFACE_SUB = era_surface.sel(latitude=REGION_LAT_SLICE, longitude=REGION_LON_SLICE)
ERA_850_SUB = era_850hPa.sel(latitude=REGION_LAT_SLICE, longitude=REGION_LON_SLICE)

print("d01_range:", d01_range)
print("Snapped to 0.25°:", region_info["snapped"])
print("Using ERA5 coord bounds:", region_info["bounds"])
print("ERA5 subdomain grid:", dict(ERA_SURFACE_SUB.sizes))


def _validate_expected_times(forecast_da, variable_name, model_name):
    valid_times = pd.to_datetime(forecast_da["valid_time"].values)
    if len(valid_times) != len(EXPECTED_VALID_TIMES) or not np.array_equal(valid_times.values, EXPECTED_VALID_TIMES.values):
        raise ValueError(
            f"Unexpected valid times for {model_name} {variable_name}: {valid_times.strftime('%Y%m%d%H').tolist()}"
        )


def compute_region_rmse_acc(model_name, variable_specs, get_forecast_and_reference_func, rmse_path, acc_path):
    base_table = pd.DataFrame(
        {
            "forecast_hour": np.arange(6, 73, 6),
            "valid_time": EXPECTED_VALID_TIMES.strftime("%Y%m%d%H"),
        }
    )
    rmse_table = base_table.copy()
    acc_table = base_table.copy()
    units = {}

    for spec in variable_specs:
        forecast_da, reference_da = get_forecast_and_reference_func(spec)
        forecast_da = forecast_da.sel(latitude=REGION_LAT_SLICE, longitude=REGION_LON_SLICE)
        reference_da = reference_da.sel(latitude=REGION_LAT_SLICE, longitude=REGION_LON_SLICE)
        forecast_da = forecast_da.transpose("valid_time", "latitude", "longitude")
        reference_da = reference_da.transpose("valid_time", "latitude", "longitude")

        _validate_expected_times(forecast_da, spec["name"], model_name)
        climatology = reference_da.mean(dim="valid_time").values
        latitude_weights = get_latitude_weights(forecast_da["latitude"].values)

        rmse_values = []
        acc_values = []
        for time_index in range(forecast_da.sizes["valid_time"]):
            forecast_field = forecast_da.isel(valid_time=time_index).values
            reference_field = reference_da.isel(valid_time=time_index).values
            rmse_values.append(weighted_rmse(forecast_field, reference_field, latitude_weights))
            acc_values.append(weighted_acc(forecast_field, reference_field, climatology, latitude_weights))

        rmse_table[spec["name"]] = rmse_values
        acc_table[spec["name"]] = acc_values
        units[spec["name"]] = spec.get("unit", "")

    rmse_table.to_csv(rmse_path, index=False)
    acc_table.to_csv(acc_path, index=False)

    print(f"\n[{model_name}] Units:")
    for variable_name, unit in units.items():
        print(f"  {variable_name}: {unit}")
    print(f"[{model_name}] Saved RMSE results to: {Path(rmse_path).resolve()}")
    print(rmse_table.round(4))
    print(f"[{model_name}] Saved ACC results to: {Path(acc_path).resolve()}")
    print(acc_table.round(4))

    return rmse_table, acc_table


# -------------------- Pangu --------------------
PANGU_SURFACE_PATH = "../../AI_forecasting_result/Ragasa/Pangu/output_48/2025-09-21-19-00to2025-09-24-19-00/surface_combined.nc"
PANGU_UPPER_PATH = "../../AI_forecasting_result/Ragasa/Pangu/output_48/2025-09-21-19-00to2025-09-24-19-00/upper_combined.nc"
PANGU_850_LEVEL = 3

if "pangu_surface" not in globals():
    pangu_surface = xr.open_dataset(PANGU_SURFACE_PATH)
if "pangu_upper" not in globals():
    pangu_upper = xr.open_dataset(PANGU_UPPER_PATH)

PANGU_VARIABLE_SPECS_REGION = [
    {"name": "U10", "layer": "surface", "var": "u10", "unit": "m s^-1"},
    {"name": "V10", "layer": "surface", "var": "v10", "unit": "m s^-1"},
    {"name": "MSLP", "layer": "surface", "var": "msl", "unit": "Pa"},
    {"name": "T2M", "layer": "surface", "var": "t2m", "unit": "K"},
    {"name": "U850", "layer": "upper", "var": "u", "unit": "m s^-1"},
    {"name": "V850", "layer": "upper", "var": "v", "unit": "m s^-1"},
    {"name": "Z850", "layer": "upper", "var": "z", "unit": "m^2 s^-2"},
    {"name": "T850", "layer": "upper", "var": "t", "unit": "K"},
 ]


def get_pangu_forecast_and_reference_region(spec):
    if spec["layer"] == "surface":
        forecast = pangu_surface[spec["var"]]
        reference = ERA_SURFACE_SUB[spec["var"]]
    else:
        forecast = pangu_upper[spec["var"]].sel(level=PANGU_850_LEVEL)
        reference = ERA_850_SUB[spec["var"]].sel(pressure_level=850, drop=True)

    forecast, reference = xr.align(forecast, reference, join="inner")
    forecast = forecast.transpose("valid_time", "latitude", "longitude")
    reference = reference.transpose("valid_time", "latitude", "longitude")
    return forecast, reference


# -------------------- GraphCast --------------------
GRAPHCAST_DIR = Path("../../AI_forecasting_result/Ragasa/Graphcast/output_48/2025092119")
GRAPHCAST_850_LEVEL = 850

GRAPHCAST_VARIABLE_SPECS_REGION = [
    {"name": "U10", "layer": "surface", "graphcast_var": "10m_u_component_of_wind", "era_var": "u10", "unit": "m s^-1"},
    {"name": "V10", "layer": "surface", "graphcast_var": "10m_v_component_of_wind", "era_var": "v10", "unit": "m s^-1"},
    {"name": "MSLP", "layer": "surface", "graphcast_var": "mean_sea_level_pressure", "era_var": "msl", "unit": "Pa"},
    {"name": "T2M", "layer": "surface", "graphcast_var": "2m_temperature", "era_var": "t2m", "unit": "K"},
    {"name": "U850", "layer": "upper", "graphcast_var": "u_component_of_wind", "era_var": "u", "unit": "m s^-1"},
    {"name": "V850", "layer": "upper", "graphcast_var": "v_component_of_wind", "era_var": "v", "unit": "m s^-1"},
    {"name": "Z850", "layer": "upper", "graphcast_var": "geopotential", "era_var": "z", "unit": "m^2 s^-2"},
    {"name": "T850", "layer": "upper", "graphcast_var": "temperature", "era_var": "t", "unit": "K"},
 ]

graphcast_files_region = sorted(
    GRAPHCAST_DIR.glob("gc_operational_predict_data_*.nc"),
    key=lambda path: int(path.stem.rsplit("_", 1)[-1]),
)
if len(graphcast_files_region) != len(EXPECTED_VALID_TIMES):
    raise ValueError(f"Expected {len(EXPECTED_VALID_TIMES)} GraphCast files, found {len(graphcast_files_region)}")


def load_graphcast_forecast_region(spec, ref_lat, ref_lon):
    forecast_slices = []
    for path in graphcast_files_region:
        with xr.open_dataset(path, decode_timedelta=False) as ds:
            valid_time = pd.Timestamp(np.asarray(ds["datetime"].values).reshape(-1)[0])
            forecast = ds[spec["graphcast_var"]]
            if "batch" in forecast.dims:
                forecast = forecast.isel(batch=0, drop=True)
            if "time" in forecast.dims:
                forecast = forecast.isel(time=0, drop=True)
            if spec["layer"] == "upper":
                forecast = forecast.sel(level=GRAPHCAST_850_LEVEL, drop=True)

            forecast = forecast.rename({"lat": "latitude", "lon": "longitude"})
            forecast = forecast.reindex(latitude=ref_lat, longitude=ref_lon)
            forecast = forecast.expand_dims(valid_time=[valid_time]).load()
            forecast_slices.append(forecast)

    return xr.concat(forecast_slices, dim="valid_time").sortby("valid_time")


def get_graphcast_forecast_and_reference_region(spec):
    if spec["layer"] == "surface":
        reference = ERA_SURFACE_SUB[spec["era_var"]].sortby("valid_time")
    else:
        reference = ERA_850_SUB[spec["era_var"]].sel(pressure_level=850, drop=True).sortby("valid_time")

    forecast = load_graphcast_forecast_region(spec, reference["latitude"], reference["longitude"])
    forecast, reference = xr.align(forecast, reference, join="inner")
    forecast = forecast.transpose("valid_time", "latitude", "longitude")
    reference = reference.transpose("valid_time", "latitude", "longitude")
    return forecast, reference


# -------------------- FengWu --------------------
FENGWU_DIR = Path("../../AI_forecasting_result/Ragasa/Fengwu/output_48/2025-09-21-13-00_to_2025-09-21-19-00")

FENGWU_VARIABLE_SPECS_REGION = [
    {"name": "U10", "layer": "surface", "fengwu_var": "u10", "era_var": "u10", "unit": "m s^-1"},
    {"name": "V10", "layer": "surface", "fengwu_var": "v10", "era_var": "v10", "unit": "m s^-1"},
    {"name": "MSLP", "layer": "surface", "fengwu_var": "msl", "era_var": "msl", "unit": "Pa"},
    {"name": "T2M", "layer": "surface", "fengwu_var": "t2m", "era_var": "t2m", "unit": "K"},
    {"name": "U850", "layer": "upper", "fengwu_var": "u850", "era_var": "u", "unit": "m s^-1"},
    {"name": "V850", "layer": "upper", "fengwu_var": "v850", "era_var": "v", "unit": "m s^-1"},
    {"name": "Z850", "layer": "upper", "fengwu_var": "z850", "era_var": "z", "unit": "m^2 s^-2"},
    {"name": "T850", "layer": "upper", "fengwu_var": "t850", "era_var": "t", "unit": "K"},
 ]

fengwu_files_region = sorted(
    FENGWU_DIR.glob("decoded_output_*.nc"),
    key=lambda path: int(path.stem.rsplit("_", 1)[-1]),
)
if len(fengwu_files_region) != len(EXPECTED_VALID_TIMES):
    raise ValueError(f"Expected {len(EXPECTED_VALID_TIMES)} Fengwu files, found {len(fengwu_files_region)}")


def load_fengwu_forecast_region(spec, ref_lat, ref_lon):
    forecast_slices = []
    for path in fengwu_files_region:
        with xr.open_dataset(path) as ds:
            forecast = ds[spec["fengwu_var"]]
            valid_times = pd.to_datetime(forecast["valid_time"].values)
            if len(valid_times) != 1:
                raise ValueError(f"Expected exactly one valid time in {path.name}, found {len(valid_times)}")

            forecast = forecast.reindex(latitude=ref_lat, longitude=ref_lon)
            forecast = forecast.load()
            forecast_slices.append(forecast)

    return xr.concat(forecast_slices, dim="valid_time").sortby("valid_time")


def get_fengwu_forecast_and_reference_region(spec):
    if spec["layer"] == "surface":
        reference = ERA_SURFACE_SUB[spec["era_var"]].sortby("valid_time")
    else:
        reference = ERA_850_SUB[spec["era_var"]].sel(pressure_level=850, drop=True).sortby("valid_time")

    forecast = load_fengwu_forecast_region(spec, reference["latitude"], reference["longitude"])
    forecast, reference = xr.align(forecast, reference, join="inner")
    forecast = forecast.transpose("valid_time", "latitude", "longitude")
    reference = reference.transpose("valid_time", "latitude", "longitude")
    return forecast, reference


# -------------------- FuXi --------------------
FUXI_DIR = Path("../../AI_forecasting_result/Ragasa/Fuxi/output_48/2025092113_to_2025092119")
FUXI_INIT_TIME = pd.Timestamp("2025-09-21 19:00")

FUXI_VARIABLE_SPECS_REGION = [
    {"name": "U10", "layer": "surface", "fuxi_level": "U10", "era_var": "u10", "unit": "m s^-1"},
    {"name": "V10", "layer": "surface", "fuxi_level": "V10", "era_var": "v10", "unit": "m s^-1"},
    {"name": "MSLP", "layer": "surface", "fuxi_level": "MSL", "era_var": "msl", "unit": "Pa"},
    {"name": "T2M", "layer": "surface", "fuxi_level": "T2M", "era_var": "t2m", "unit": "K"},
    {"name": "U850", "layer": "upper", "fuxi_level": "U850", "era_var": "u", "unit": "m s^-1"},
    {"name": "V850", "layer": "upper", "fuxi_level": "V850", "era_var": "v", "unit": "m s^-1"},
    {"name": "Z850", "layer": "upper", "fuxi_level": "Z850", "era_var": "z", "unit": "m^2 s^-2"},
    {"name": "T850", "layer": "upper", "fuxi_level": "T850", "era_var": "t", "unit": "K"},
 ]

fuxi_files_region = sorted(FUXI_DIR.glob("*.nc"), key=lambda path: int(path.stem))
if len(fuxi_files_region) != len(EXPECTED_VALID_TIMES):
    raise ValueError(f"Expected {len(EXPECTED_VALID_TIMES)} Fuxi files, found {len(fuxi_files_region)}")


def load_fuxi_forecast_region(spec, ref_lat, ref_lon):
    forecast_slices = []
    for path in fuxi_files_region:
        lead_hour = int(path.stem)
        valid_time = FUXI_INIT_TIME + pd.Timedelta(hours=lead_hour)
        with xr.open_dataset(path, decode_timedelta=False) as ds:
            forecast = ds["__xarray_dataarray_variable__"].sel(level=spec["fuxi_level"])
            if "time" in forecast.dims:
                forecast = forecast.isel(time=0, drop=True)
            if "step" in forecast.dims:
                forecast = forecast.isel(step=0, drop=True)

            forecast = forecast.rename({"lat": "latitude", "lon": "longitude"})
            forecast = forecast.reindex(latitude=ref_lat, longitude=ref_lon)
            forecast = forecast.expand_dims(valid_time=[valid_time]).load()
            forecast_slices.append(forecast)

    return xr.concat(forecast_slices, dim="valid_time").sortby("valid_time")


def get_fuxi_forecast_and_reference_region(spec):
    if spec["layer"] == "surface":
        reference = ERA_SURFACE_SUB[spec["era_var"]].sortby("valid_time")
    else:
        reference = ERA_850_SUB[spec["era_var"]].sel(pressure_level=850, drop=True).sortby("valid_time")

    forecast = load_fuxi_forecast_region(spec, reference["latitude"], reference["longitude"])
    forecast, reference = xr.align(forecast, reference, join="inner")
    forecast = forecast.transpose("valid_time", "latitude", "longitude")
    reference = reference.transpose("valid_time", "latitude", "longitude")
    return forecast, reference


# -------------------- Aurora --------------------
AURORA_DIR = Path("../../AI_forecasting_result/Ragasa/Aurora/output_48/WP/Ragasa/202509211300")

AURORA_VARIABLE_SPECS_REGION = [
    {"name": "U10", "layer": "surface", "aurora_var": "u10", "era_var": "u10", "unit": "m s^-1"},
    {"name": "V10", "layer": "surface", "aurora_var": "v10", "era_var": "v10", "unit": "m s^-1"},
    {"name": "MSLP", "layer": "surface", "aurora_var": "msl", "era_var": "msl", "unit": "Pa"},
    {"name": "T2M", "layer": "surface", "aurora_var": "t2m", "era_var": "t2m", "unit": "K"},
    {"name": "U850", "layer": "upper", "aurora_var": "u", "era_var": "u", "unit": "m s^-1"},
    {"name": "V850", "layer": "upper", "aurora_var": "v", "era_var": "v", "unit": "m s^-1"},
    {"name": "Z850", "layer": "upper", "aurora_var": "z", "era_var": "z", "unit": "m^2 s^-2"},
    {"name": "T850", "layer": "upper", "aurora_var": "t", "era_var": "t", "unit": "K"},
 ]

aurora_files_region = sorted(AURORA_DIR.glob("*.nc"))
if len(aurora_files_region) != len(EXPECTED_VALID_TIMES):
    raise ValueError(f"Expected {len(EXPECTED_VALID_TIMES)} Aurora files, found {len(aurora_files_region)}")


def load_aurora_forecast_region(spec, ref_lat, ref_lon):
    forecast_slices = []
    for path in aurora_files_region:
        with xr.open_dataset(path, decode_timedelta=False) as ds:
            valid_time = pd.Timestamp(np.asarray(ds["time"].values).reshape(-1)[0])
            forecast = ds[spec["aurora_var"]]
            if spec["layer"] == "upper":
                forecast = forecast.sel(level=850, drop=True)
            if "time" in forecast.dims:
                forecast = forecast.isel(time=0, drop=True)

            forecast = forecast.rename({"lat": "latitude", "lon": "longitude"})
            forecast = forecast.reindex(latitude=ref_lat, longitude=ref_lon)
            forecast = forecast.expand_dims(valid_time=[valid_time]).load()
            forecast_slices.append(forecast)

    return xr.concat(forecast_slices, dim="valid_time").sortby("valid_time")


def get_aurora_forecast_and_reference_region(spec):
    if spec["layer"] == "surface":
        reference = ERA_SURFACE_SUB[spec["era_var"]].sortby("valid_time")
    else:
        reference = ERA_850_SUB[spec["era_var"]].sel(pressure_level=850, drop=True).sortby("valid_time")

    forecast = load_aurora_forecast_region(spec, reference["latitude"], reference["longitude"])
    forecast, reference = xr.align(forecast, reference, join="inner")
    forecast = forecast.transpose("valid_time", "latitude", "longitude")
    reference = reference.transpose("valid_time", "latitude", "longitude")
    return forecast, reference


# -------------------- Run region calculations & save --------------------
outputs = {
    "pangu": {
        "specs": PANGU_VARIABLE_SPECS_REGION,
        "getter": get_pangu_forecast_and_reference_region,
    },
    "graphcast": {
        "specs": GRAPHCAST_VARIABLE_SPECS_REGION,
        "getter": get_graphcast_forecast_and_reference_region,
    },
    "fengwu": {
        "specs": FENGWU_VARIABLE_SPECS_REGION,
        "getter": get_fengwu_forecast_and_reference_region,
    },
    "fuxi": {
        "specs": FUXI_VARIABLE_SPECS_REGION,
        "getter": get_fuxi_forecast_and_reference_region,
    },
    "aurora": {
        "specs": AURORA_VARIABLE_SPECS_REGION,
        "getter": get_aurora_forecast_and_reference_region,
    },
}

for model_key, cfg in outputs.items():
    rmse_out = OUTPUT_DIR / f"{model_key}_era5_rmse_{REGION_TAG}.csv"
    acc_out = OUTPUT_DIR / f"{model_key}_era5_acc_{REGION_TAG}.csv"
    compute_region_rmse_acc(model_key, cfg["specs"], cfg["getter"], rmse_out, acc_out)

d01_range: (9.733, 33.868, 97.149, 131.277)
Snapped to 0.25°: (9.5, 34.0, 97.0, 131.5)
Using ERA5 coord bounds: (9.5, 34.0, 97.0, 131.5)
ERA5 subdomain grid: {'valid_time': 12, 'latitude': 99, 'longitude': 139}

[pangu] Units:
  U10: m s^-1
  V10: m s^-1
  MSLP: Pa
  T2M: K
  U850: m s^-1
  V850: m s^-1
  Z850: m^2 s^-2
  T850: K
[pangu] Saved RMSE results to: E:\CityU\Paper Code\06_AI_WRF_UCM\Figs\Fig3\data\pangu_era5_rmse_d01.csv
    forecast_hour  valid_time     U10     V10      MSLP     T2M    U850  \
0               6  2025092201  1.0308  0.8237   57.6435  0.5345  1.6957   
1              12  2025092207  1.1000  0.9271   65.9436  0.8798  1.9007   
2              18  2025092213  1.3265  1.1966   89.2899  0.7835  2.1003   
3              24  2025092219  1.2828  1.3791   88.5678  0.6521  2.2093   
4              30  2025092301  1.2430  1.4842  102.2224  0.7273  2.2861   
5              36  2025092307  1.6037  1.6581  121.2204  1.0178  2.7756   
6              42  2025092313  1.9774  